## 8 Ball Table Analyses - Task 1 Computer Vision

In [ ]:
import os
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import math
import colorsys


In [ ]:
IMG_DIR = Path("development_set/")
OUTPUT_DIR= Path("output/top_views")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_EXTENSIONS = {".jpg",".jpeg", ".png", ".bmp"}
image_paths = sorted([ path for path in IMG_DIR.iterdir()
                       if path.suffix.lower() in VALID_EXTENSIONS])

print(f"Found {len(image_paths)} images")
for i, path in enumerate(image_paths[:5]):
    print(f"[{i}] {path.name}")


In [ ]:
def show_images_grid(images, titles=None, figsize_per_row=(18, 5)):
    if isinstance(images, np.ndarray):
        images = [images]

    if isinstance(titles, str):
        titles = [titles]

    num_images = len(images)

    if num_images == 0:
        print("No images provided to display.")
        return

    cols = min(3, num_images)

    rows = math.ceil(num_images / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(figsize_per_row[0], figsize_per_row[1] * rows))

    if hasattr(axes, 'flatten'):
        axes = axes.flatten()
    else:
        axes = [axes]

    for i, ax in enumerate(axes):
        if i < num_images:
            img = images[i]

            if len(img.shape) == 3:
                img_to_show = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                ax.imshow(img_to_show)
            else:
                ax.imshow(img, cmap="gray")

            if titles and i < len(titles):
                ax.set_title(titles[i])
            else:
                ax.set_title(f"Image {i}")

            ax.axis("off")

        else:
            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
if not image_paths:
    raise FileNotFoundError(f"No image files found in {IMG_DIR}")

paths_to_load = image_paths[:18]

loaded_images = []
image_titles = []

for path in paths_to_load:
    img = cv2.imread(str(path))

    if img is None:
        print(f"Could not load image: {path}")
        continue

    loaded_images.append(img)
    image_titles.append(f"Original: {path.name}")

    print(f"Loaded: {path.name} | Shape: {img.shape}")

if not loaded_images:
    raise FileNotFoundError("Failed to load any of the selected images.")

show_images_grid(loaded_images, titles=image_titles)

# Top View of table

In [ ]:
# Each tuple is (H, S, V) in OpenCV HSV space (H: 0-179, S: 0-255, V: 0-255)
TARGET_HSV_LIST = [
    (102, 140, 0),   # primary green felt
]


In [ ]:
def isolate_table_color(image, target_hsv_list=TARGET_HSV_LIST, tol_h=5, tol_s=60):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    combined = np.zeros(hsv.shape[:2], dtype=np.uint8)
    for target_hsv in target_hsv_list:
        h, s, _ = target_hsv

        lower = np.array([max(h - tol_h, 0),
                          max(s - tol_s, 0),
                          40],
                         dtype=np.uint8)

        upper = np.array([min(h + tol_h, 179),
                          min(s + tol_s, 255),
                          255],
                         dtype=np.uint8)

        combined = cv2.bitwise_or(combined, cv2.inRange(hsv, lower, upper))

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    combined = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kernel)
    combined = cv2.morphologyEx(combined, cv2.MORPH_OPEN,  kernel)
    return combined


In [ ]:
table_masks = []

for img in loaded_images:
    single_mask = isolate_table_color(img)

    table_masks.append(single_mask)

show_images_grid(table_masks, image_titles)

In [ ]:
def show_images_mask_grid(images, masks, titles=None, figsize_per_row=(18, 5)):
    num_pairs = len(images)
    total_images = num_pairs * 2

    cols = min(3, total_images)
    rows = math.ceil(total_images / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(20, 4* rows))

    if hasattr(axes, 'flatten'):
        axes = axes.flatten()
    else:
        axes = [axes]

    for i in range(num_pairs):
        orig_idx = i * 2
        mask_idx = i * 2 + 1

        img = images[i]
        if len(img.shape) == 3:
            img_to_show = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[orig_idx].imshow(img_to_show)
        else:
            axes[orig_idx].imshow(img, cmap="gray")

        axes[orig_idx].set_title(f"Original {i}")
        axes[orig_idx].axis("off")

        mask = masks[i]
        axes[mask_idx].imshow(mask, cmap="gray")
        axes[mask_idx].set_title(f"Mask {i}")
        axes[mask_idx].axis("off")

    for j in range(total_images, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()

show_images_mask_grid(loaded_images, table_masks, image_titles)

In [ ]:
def get_table_contour(binary_mask):
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        print("No contours found in the mask!")
        return None

    table_contour = max(contours, key=cv2.contourArea)
    return table_contour

In [ ]:
contour_images = []
contour_titles = []
my_contours = []

for i, (img, mask) in enumerate(zip(loaded_images, table_masks)):
    contour_img = img.copy()
    table_contour = get_table_contour(mask)

    my_contours.append(table_contour)

    if table_contour is not None:
        cv2.drawContours(contour_img, [table_contour], -1, (0, 255, 0), 3)
        contour_titles.append(f"Table Contour {i+1}")
    else:
        contour_titles.append(f"No Contour {i+1}")

    contour_images.append(contour_img)

show_images_grid(contour_images, titles=contour_titles)

In [ ]:
def order_points(pts):
    pts = np.array(pts, dtype="float32")
    center = np.mean(pts, axis=0)

    angles = np.arctan2(pts[:, 1] - center[1], pts[:, 0] - center[0])
    pts = pts[np.argsort(angles)]

    s = pts.sum(axis=1)
    start = np.argmin(s)
    pts = np.roll(pts, -start, axis=0)

    return pts

In [ ]:
def get_table_corners(table_contour, padding=0):
    hull = cv2.convexHull(table_contour)
    peri = cv2.arcLength(hull, True)

    pts = None
    for eps in np.arange(0.005, 0.10, 0.002):
        approx = cv2.approxPolyDP(hull, eps * peri, True)
        if len(approx) == 4:
            pts = approx.reshape(4, 2).astype("float32")
            break

    if pts is None:
        rect = cv2.minAreaRect(hull)
        pts = cv2.boxPoints(rect).astype("float32")

    ordered = order_points(pts)

    if padding > 0:
        center = np.mean(ordered, axis=0)
        padded = []
        for pt in ordered:
            direction = pt - center
            unit_direction = direction / np.linalg.norm(direction)
            padded.append(pt + unit_direction * padding)
        return np.array(padded, dtype="float32")

    return ordered

In [ ]:
corner_images = []
corner_titles = []
my_corners = []

for i, (img, contour) in enumerate(zip(loaded_images, my_contours)):
    corner_img = img.copy()

    if contour is not None:
        ordered_box = get_table_corners(contour)
        my_corners.append(ordered_box)


        draw_box = np.intp(ordered_box)
        cv2.drawContours(corner_img, [draw_box], -1, (0, 0, 255), 3)

        corner_titles.append(f"Padded Corners {i+1}")
    else:
        my_corners.append(None)
        corner_titles.append(f"No Table {i+1}")

    corner_images.append(corner_img)

show_images_grid(corner_images, titles=corner_titles)

In [ ]:
def get_top_view(image, src_corners, width=1000, height=500):
    (tl, tr, br, bl) = src_corners

    width_top = np.linalg.norm(tr - tl)
    height_left = np.linalg.norm(bl - tl)

    if height_left > width_top:
        src_corners = np.array([bl, tl, tr, br], dtype="float32")

    dst_corners = np.array([
        [0, 0],
        [width - 1, 0],
        [width - 1, height - 1],
        [0, height - 1]
    ], dtype="float32")

    M = cv2.getPerspectiveTransform(src_corners, dst_corners)
    top_view = cv2.warpPerspective(image, M, (width, height))

    return top_view

In [ ]:
top_view_images = []
top_view_titles = []

for i, (img, corners) in enumerate(zip(loaded_images, my_corners)):

    if corners is not None:
        flat_table = get_top_view(img, corners)

        top_view_images.append(flat_table)
        top_view_titles.append(f"Top View {i+1}")
    else:
        top_view_images.append(np.zeros((600, 800, 3), dtype=np.uint8))
        top_view_titles.append(f"Failed {i+1}")

show_images_grid(top_view_images, titles=top_view_titles)

# Get the balls

In [ ]:
initial_view_masks = []

for img in top_view_images:
    single_mask = isolate_table_color(img)


    initial_view_masks.append(single_mask)

show_images_mask_grid(top_view_images, initial_view_masks, image_titles)

In [ ]:
felt_masks = []

for mask in initial_view_masks:
    # Fill holes in the raw felt mask — no erosion, keeps balls near cushions
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled = np.zeros_like(mask)
    if contours:
        largest = max(contours, key=cv2.contourArea)
        cv2.drawContours(filled, [largest], -1, 255, cv2.FILLED)
    felt_masks.append(filled)

show_images_mask_grid(top_view_images, felt_masks, image_titles)


# Hough Circles

- Use this video to understand it: https://www.youtube.com/watch?v=Ltqt24SQQoI
- Review CLAHE: https://www.youtube.com/watch?v=tn2kmbUVK50

In [ ]:
def detect_balls(image, felt_mask, min_radius=16, max_radius=30, min_dist=30, param1=50, param2=14):
    # Convert full image to grayscale — no mask applied so no fake boundary edges
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (9, 9), 2)

    circles = cv2.HoughCircles(
        blurred,
        cv2.HOUGH_GRADIENT,
        dp=1,
        minDist=min_dist,
        param1=param1,   # Canny high threshold (used internally)
        param2=param2,   # accumulator threshold
        minRadius=min_radius,
        maxRadius=max_radius,
    )

    if circles is None:
        return []

    # Filter: only keep circles whose center is inside the felt mask
    result = []
    for x, y, r in np.round(circles[0]).astype(int):
        if 0 <= y < felt_mask.shape[0] and 0 <= x < felt_mask.shape[1]:
            if felt_mask[y, x] > 0:
                result.append((int(x), int(y), int(r)))
    return result


# Debug Hough Circles

In [ ]:
# %matplotlib widget
# import sys
# from IPython.display import display
# import ipywidgets as widgets

# fig, ax = plt.subplots(figsize=(14, 7))
# debug_img = top_view_images[0]
# gray_d  = cv2.cvtColor(debug_img, cv2.COLOR_BGR2GRAY)
# blur_d  = cv2.GaussianBlur(gray_d, (9, 9), 2)
# canny_d = cv2.Canny(blur_d, 25, 50)
# ax.imshow(canny_d, cmap="gray")
# ax.set_title("Click LEFT edge then RIGHT edge of a ball")

# out = widgets.Output()
# display(out)

# clicks = []
# annotations = []

# def on_click(event):
#     if event.inaxes != ax:
#         return
#     clicks.append((event.xdata, event.ydata))
#     ax.plot(event.xdata, event.ydata, "r+", markersize=12)

#     if len(clicks) % 2 == 0:
#         x1, y1 = clicks[-2]
#         x2, y2 = clicks[-1]
#         dist = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)
#         radius = dist / 2
#         # Draw line and label on plot
#         ax.plot([x1, x2], [y1, y2], "r-", linewidth=1.5)
#         mx, my = (x1 + x2) / 2, (y1 + y2) / 2
#         ann = ax.annotate(
#             f"r={radius:.1f}px",
#             xy=(mx, my), fontsize=9, color="yellow",
#             bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.6),
#         )
#         annotations.append(ann)
#         fig.canvas.draw()
#         with out:
#             print(
#                 f"Diameter: {dist:.1f}px  →  radius: {radius:.1f}px  |  "
#                 f"Suggested: min_radius={int(radius*0.8)}, max_radius={int(radius*1.2)}",
#                 flush=True,
#             )
#     else:
#         fig.canvas.draw()

# fig.canvas.mpl_connect("button_press_event", on_click)
# plt.tight_layout()
# plt.show()


In [ ]:
PARAM1 = 50

debug_img = top_view_images[0]
gray_d   = cv2.cvtColor(debug_img, cv2.COLOR_BGR2GRAY)
blur_d   = cv2.GaussianBlur(gray_d, (9, 9), 2)
canny_d  = cv2.Canny(blur_d, PARAM1 // 2, PARAM1)

# Also show masked vs unmasked grayscale for comparison
masked_gray = cv2.bitwise_and(gray_d, gray_d, mask=felt_masks[0])
masked_canny = cv2.Canny(cv2.GaussianBlur(masked_gray, (9, 9), 2), PARAM1 // 2, PARAM1)

show_images_grid(
    [debug_img, gray_d, blur_d, canny_d, masked_gray, masked_canny],
    titles=[
        "Top View", "Grayscale", "Blurred",
        "Canny (no mask) — what Hough sees",
        "Masked Grayscale", "Canny (masked) — fake boundary edges visible",
    ]
)


In [ ]:
import ipywidgets as widgets
from IPython.display import display
from IPython.display import Image as IPImage

IMG_IDX = 0  # change to test a different image

tuner_img  = top_view_images[IMG_IDX]
tuner_mask = felt_masks[IMG_IDX]
gray_t = cv2.cvtColor(tuner_img, cv2.COLOR_BGR2GRAY)
blur_t = cv2.GaussianBlur(gray_t, (9, 9), 2)

out = widgets.Output()

w_min_r    = widgets.IntSlider(value=16,  min=1,   max=100, step=1,  description="min_radius", style={"description_width": "initial"}, layout=widgets.Layout(width="450px"))
w_max_r    = widgets.IntSlider(value=40,  min=5,   max=150, step=1,  description="max_radius", style={"description_width": "initial"}, layout=widgets.Layout(width="450px"))
w_min_dist = widgets.IntSlider(value=30,  min=5,   max=200, step=1,  description="min_dist",   style={"description_width": "initial"}, layout=widgets.Layout(width="450px"))
w_p1       = widgets.IntSlider(value=50,  min=10,  max=300, step=5,  description="param1",     style={"description_width": "initial"}, layout=widgets.Layout(width="450px"))
w_p2       = widgets.IntSlider(value=14,  min=1,   max=100, step=1,  description="param2",     style={"description_width": "initial"}, layout=widgets.Layout(width="450px"))
w_dp       = widgets.IntSlider(value=10,  min=5,   max=30,  step=1,  description="dp (x0.1)",  style={"description_width": "initial"}, layout=widgets.Layout(width="450px"))

def render(_=None):
    min_r    = w_min_r.value
    max_r    = w_max_r.value
    min_dist = w_min_dist.value
    p1       = w_p1.value
    p2       = w_p2.value
    dp       = w_dp.value

    canny_preview = cv2.Canny(blur_t, p1 // 2, p1)

    raw = cv2.HoughCircles(
        blur_t, cv2.HOUGH_GRADIENT,
        dp=dp / 10,
        minDist=min_dist,
        param1=p1,
        param2=p2,
        minRadius=min_r,
        maxRadius=max_r,
    )

    vis = tuner_img.copy()
    count = 0
    if raw is not None:
        for x, y, r in np.round(raw[0]).astype(int):
            xi, yi = int(x), int(y)
            if 0 <= yi < tuner_mask.shape[0] and 0 <= xi < tuner_mask.shape[1]:
                if tuner_mask[yi, xi] > 0:
                    cv2.circle(vis, (xi, yi), r, (0, 255, 0), 2)
                    cv2.circle(vis, (xi, yi), 2, (0, 0, 255), 3)
                    count += 1

    canny_bgr = cv2.cvtColor(canny_preview, cv2.COLOR_GRAY2BGR)
    combined  = np.hstack([canny_bgr, vis])
    label = f"Canny low={p1//2} high={p1}  |  Detections: {count}  dp={dp/10:.1f}  minDist={min_dist}  p1={p1}  p2={p2}  r=[{min_r},{max_r}]"
    cv2.putText(combined, label, (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

    _, buf = cv2.imencode(".png", combined)
    with out:
        out.clear_output(wait=True)
        display(IPImage(data=buf.tobytes()))

for w in [w_min_r, w_max_r, w_min_dist, w_p1, w_p2, w_dp]:
    w.observe(render, names="value")

ui = widgets.VBox([
    widgets.HBox([w_min_r, w_max_r]),
    widgets.HBox([w_min_dist, w_dp]),
    widgets.HBox([w_p1, w_p2]),
    out,
])

display(ui)
render()  # draw initial image


In [ ]:
ball_detections = []
detection_images = []

for img, felt_mask in zip(top_view_images, felt_masks):
    circles = detect_balls(img, felt_mask)
    ball_detections.append(circles)

    vis = img.copy()
    for (x, y, r) in circles:
        cv2.circle(vis, (x, y), r, (0, 255, 0), 2)
        cv2.circle(vis, (x, y), 2, (0, 0, 255), 3)
    detection_images.append(vis)

    print(f"Detected {len(circles)} balls")

show_images_grid(detection_images, top_view_titles)
